In [1]:
import pandas as pd
import numpy as np
import re
import ast
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from pathlib import Path
from sklearn.preprocessing import StandardScaler

RELATIVE_FEATURES_DIR = '../../data/features'
LANGUAGES = [                               # Languages to consider
    'ar',
    'de',
    'en',
    'es',
    'ja',
    'pl',
    'ru',
    'sw',
    'tr',
    'uz',
    'yue',
    'zh-TW'
]
NO_OF_LANGUAGES = len(LANGUAGES)

In [2]:
def fix_array_format(arr_str):
    arr_str = re.sub(r"\s+", ",", arr_str.strip())
    arr_str = arr_str.replace("[,", "[").replace(",]", "]")
    return np.array(ast.literal_eval(arr_str))

def fix_list_values(df):
    list_columns = ['mfcc_mean', 'mfcc_var', 'spectral_contrast_mean', 'spectral_contrast_var', 'chroma_mean', 'chroma_var']
    for col in list_columns:
        features_df = df[col].apply(fix_array_format).apply(pd.Series)
        features_df.columns = [f'{col}_{i}' for i in range(features_df.shape[1])]
        df = pd.concat([df.drop(columns=[col]), features_df], axis=1)
    
    return df

In [12]:
# we need a dataset containing all the features of all languages, so array [x, 76] where x is the number of samples
all_languages_features = []
y = []

for language in LANGUAGES:
    file = Path(f'{RELATIVE_FEATURES_DIR}') / f'{language}_features.csv'
    scaler = StandardScaler()
    df = pd.read_csv(file)

    # select only male speakers
    df = df[df['gender'].str.startswith('male')]
    # filter out teens
    df = df[~df['age'].str.contains('teens', case=False)]
    # drop columns starting with zrc_
    df = df.loc[:, ~df.columns.str.startswith('zcr_')]
    df = df.iloc[:, 3:] # drop non-numeric columns
    df = fix_list_values(df)
    X = df.to_numpy(dtype=float)

    scaled_X = scaler.fit_transform(X)
    all_languages_features.append(scaled_X)
    y.extend([file.stem.split('_')[0]] * scaled_X.shape[0])
    
# merge all languages features into one array
all_languages_features = np.concatenate(all_languages_features, axis=0)

### RFE (recursive feature elimination)

In [15]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

all_languages_features_df = pd.DataFrame(all_languages_features)

# Handle missing values
imputer = SimpleImputer(strategy='mean')
all_languages_features_df = pd.DataFrame(imputer.fit_transform(all_languages_features_df))

# Initialize the 
model = LogisticRegression()

# Perform RFE
rfe = RFE(model, n_features_to_select=3)
fit = rfe.fit(all_languages_features_df, y)

# Get selected features
selected_features = all_languages_features_df.columns[fit.support_]
selected_features = all_languages_features_df[selected_features]

In [16]:
print(selected_features)

              55        56        57
0       1.041847  1.627207  1.991438
1       0.487322  0.126369 -0.274110
2      -0.131183 -0.198555  0.473482
3      -0.058246  0.081338 -0.665096
4      -0.295008 -0.156682  0.267378
...          ...       ...       ...
936069 -0.107201 -0.266262  0.563256
936070 -0.545105 -0.145368  0.204931
936071  0.111281  0.023423  0.470816
936072 -0.310753  0.190253  1.249990
936073 -0.920570 -0.547499  0.452011

[936074 rows x 3 columns]


### Wzajemna informacja (Mutual Information)

In [30]:
from sklearn.feature_selection import mutual_info_classif
mi_scores = mutual_info_classif(X, y)

In [36]:
print(np.argsort(mi_scores)[:3])

[58 45 34]


In [40]:
print(df.columns[58])
print(df.columns[45])
print(df.columns[34])

chroma_mean_8
spectral_contrast_var_2
mfcc_var_11


In [62]:
print(df.columns[np.argsort(mi_scores)])

Index(['chroma_mean_8', 'spectral_contrast_var_2', 'mfcc_var_11',
       'chroma_mean_9', 'chroma_mean_10', 'mfcc_mean_4', 'mfcc_mean_12',
       'chroma_var_8', 'spectral_contrast_mean_1', 'chroma_mean_11',
       'mfcc_var_12', 'chroma_var_9', 'chroma_mean_7', 'mfcc_var_10',
       'chroma_var_10', 'mfcc_var_8', 'chroma_var_7',
       'spectral_contrast_mean_3', 'chroma_var_1', 'spectral_contrast_var_4',
       'spectral_contrast_var_3', 'chroma_var_0', 'chroma_mean_0',
       'mfcc_mean_10', 'chroma_mean_1', 'mfcc_var_4', 'mfcc_mean_6',
       'mfcc_mean_1', 'mfcc_var_3', 'chroma_mean_6', 'spectral_contrast_var_5',
       'mfcc_mean_11', 'chroma_var_6', 'mfcc_mean_8', 'mfcc_var_6',
       'mfcc_var_5', 'mfcc_var_9', 'spectral_contrast_var_1', 'mfcc_var_7',
       'chroma_var_3', 'chroma_var_4', 'chroma_var_2', 'mfcc_mean_2',
       'chroma_var_11', 'spectral_contrast_var_0', 'spectral_contrast_mean_2',
       'chroma_var_5', 'spectral_contrast_mean_4', 'chroma_mean_2',
       'mfcc_

### Feature Importance z modeli uczących się


#### Las losowy (Random Forest)

In [51]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X, y)
importances = model.feature_importances_

In [52]:
print(np.argsort(importances)[:3])

[70 62 69]


In [59]:
print(df.columns[70])
print(df.columns[62])
print(df.columns[69])

chroma_var_8
chroma_var_0
chroma_var_7


In [63]:
print(df.columns[np.argsort(importances)])

Index(['chroma_var_8', 'chroma_var_0', 'chroma_var_7', 'mfcc_var_11',
       'spectral_contrast_var_3', 'chroma_var_1', 'chroma_var_10',
       'mfcc_var_12', 'chroma_mean_8', 'chroma_var_9', 'chroma_mean_7',
       'chroma_mean_0', 'mfcc_var_9', 'mfcc_var_10', 'chroma_var_6',
       'mfcc_var_8', 'spectral_contrast_var_2', 'chroma_var_2', 'mfcc_var_4',
       'chroma_var_3', 'chroma_var_4', 'chroma_var_5', 'chroma_mean_10',
       'chroma_mean_11', 'chroma_mean_9', 'spectral_contrast_var_4',
       'chroma_var_11', 'mfcc_var_5', 'spectral_contrast_mean_3',
       'chroma_mean_1', 'chroma_mean_6', 'mfcc_mean_12', 'mfcc_var_7',
       'spectral_contrast_var_1', 'spectral_contrast_var_5',
       'spectral_contrast_mean_1', 'mfcc_mean_11', 'mfcc_var_3',
       'chroma_mean_4', 'chroma_mean_5', 'spectral_contrast_mean_2',
       'spectral_contrast_mean_4', 'chroma_mean_3', 'mfcc_var_6',
       'mfcc_mean_10', 'spectral_contrast_mean_5', 'chroma_mean_2',
       'mfcc_mean_4', 'mfcc_mean_8',

#### Gradient Boosting 

In [56]:
from xgboost import XGBClassifier
y_numeric = [LANGUAGES.index(language) for language in y]
model = XGBClassifier()
model.fit(X, y_numeric)
importances_xgboost = model.feature_importances_

In [57]:
print(np.argsort(importances_xgboost)[:3])

[63 70 34]


In [58]:
print(df.columns[63])
print(df.columns[70])
print(df.columns[34])

chroma_var_1
chroma_var_8
mfcc_var_11


In [73]:
importances_xgboost.shape

(74,)

In [74]:
print(df.columns[np.argsort(importances_xgboost)])

Index(['chroma_var_1', 'chroma_var_8', 'mfcc_var_11',
       'spectral_contrast_var_3', 'chroma_var_7', 'chroma_var_9',
       'chroma_var_10', 'mfcc_var_10', 'chroma_var_3', 'chroma_var_2',
       'chroma_var_0', 'chroma_mean_7', 'chroma_var_6', 'mfcc_var_8',
       'chroma_var_4', 'chroma_var_5', 'chroma_mean_8', 'chroma_mean_11',
       'spectral_contrast_var_2', 'mfcc_var_7', 'chroma_mean_6', 'mfcc_var_12',
       'chroma_mean_1', 'spectral_contrast_var_4', 'chroma_var_11',
       'chroma_mean_9', 'mfcc_var_4', 'spectral_contrast_mean_3', 'mfcc_var_9',
       'spectral_contrast_mean_1', 'mfcc_mean_2', 'mfcc_var_5',
       'spectral_contrast_var_5', 'chroma_mean_4', 'spectral_contrast_mean_5',
       'spectral_contrast_var_1', 'chroma_mean_10', 'spectral_centroid_var',
       'spectral_contrast_mean_2', 'mfcc_mean_8', 'mfcc_mean_4',
       'mfcc_mean_11', 'chroma_mean_3', 'mfcc_mean_12', 'chroma_mean_5',
       'chroma_mean_2', 'spectral_centroid_mean', 'mfcc_mean_10',
       'spect

###  Selekcja cech oparta na L1 (LASSO)

In [66]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(penalty='l1', solver='liblinear')
model.fit(X, y)
importance_lasso = model.coef_

In [67]:
print(np.argsort(importance_lasso)[0][:3])

[ 0 52 51]


In [68]:
print(df.columns[np.argsort(importance_lasso)[0]])

Index(['pitch_mean', 'chroma_mean_2', 'chroma_mean_1', 'chroma_mean_0',
       'spectral_contrast_var_6', 'spectral_contrast_var_5',
       'spectral_contrast_var_4', 'chroma_mean_3', 'spectral_contrast_var_3',
       'spectral_contrast_var_1', 'spectral_contrast_var_0',
       'spectral_contrast_mean_6', 'spectral_contrast_mean_5',
       'spectral_contrast_mean_4', 'spectral_contrast_mean_3',
       'spectral_contrast_var_2', 'chroma_mean_4', 'chroma_mean_5',
       'chroma_mean_6', 'chroma_var_9', 'chroma_var_8', 'chroma_var_7',
       'chroma_var_6', 'chroma_var_5', 'chroma_var_4', 'chroma_var_3',
       'chroma_var_2', 'chroma_var_1', 'chroma_var_0', 'chroma_mean_11',
       'chroma_mean_10', 'chroma_mean_9', 'chroma_mean_8', 'chroma_mean_7',
       'spectral_contrast_mean_2', 'spectral_contrast_mean_1',
       'spectral_contrast_mean_0', 'mfcc_var_12', 'mfcc_mean_5', 'mfcc_mean_4',
       'mfcc_mean_3', 'mfcc_mean_2', 'mfcc_mean_1', 'mfcc_mean_0',
       'spectral_flatness_var', 

### Analiza komponentów liniowych

In [69]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
lda = c()
lda.fit(X, y)

LinearDiscriminantAnalysis()

In [70]:
lda_coefficients = lda.coef_
feature_importance = np.linalg.norm(lda_coefficients, axis=0)
feature_ranking = np.argsort(feature_importance)[::-1]  # Indeksy cech od najbardziej do najmniej istotnych
ranked_features = [df.columns[i] for i in feature_ranking]

In [71]:
print(ranked_features)

['chroma_mean_5', 'chroma_mean_6', 'chroma_mean_4', 'chroma_mean_3', 'chroma_mean_9', 'chroma_mean_10', 'chroma_mean_7', 'chroma_mean_0', 'chroma_mean_11', 'chroma_mean_1', 'chroma_mean_2', 'chroma_mean_8', 'spectral_bandwidth_mean', 'chroma_var_5', 'chroma_var_6', 'spectral_centroid_mean', 'chroma_var_4', 'chroma_var_0', 'chroma_var_9', 'chroma_var_10', 'chroma_var_3', 'spectral_contrast_mean_6', 'chroma_var_1', 'chroma_var_11', 'chroma_var_7', 'chroma_var_2', 'mfcc_var_1', 'spectral_contrast_mean_3', 'spectral_bandwidth_var', 'spectral_contrast_mean_1', 'chroma_var_8', 'spectral_flatness_mean', 'pitch_mean', 'mfcc_var_0', 'pitch_var', 'spectral_contrast_mean_4', 'spectral_flatness_var', 'spectral_contrast_mean_2', 'spectral_centroid_var', 'mfcc_mean_5', 'spectral_contrast_var_3', 'spectral_contrast_mean_5', 'mfcc_mean_3', 'spectral_contrast_var_6', 'mfcc_mean_2', 'spectral_contrast_var_1', 'mfcc_mean_1', 'spectral_contrast_var_4', 'mfcc_mean_6', 'mfcc_var_2', 'mfcc_mean_0', 'mfcc_var